In [ ]:
"""
session_summary.ipynb

plot model metrics across sessions

Author: Stellina X. Ao
Created: 2026-05-04
Last Modified: 2026-05-06
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
from utils.paths import FIGURES_DIR

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

## LVM - one latent

In [ ]:
"""
ALL & PER-REGION
R2 of encoding and latent variable model 
Improvement over task variable model and QI
Single latent histograms
Quantify time scale of latents (PSD)
"""

In [ ]:
from utils.paths import MODELS_DIR
import pickle
import numpy as np
from core.data import subject_ids, session_ids, colors_subj


def get_res_tv_lvms(subj_id, regions=["all", "ACC", "M2", "DMS", "DLS"]):
    subj_idx = np.where(subject_ids == subj_id)[0][0]

    res_tv_lvms = []
    for sess_id in session_ids[subj_idx]:
        file_path = (
            MODELS_DIR / "fit" / subj_id / sess_id / "one_latent" / "results_dict.pkl"
        )

        with open(file_path, "rb") as f:
            res_dict = pickle.load(f)

        res_tv_lvm = {}
        for region in regions:
            try:
                res_tv_lvms_ = res_dict[region]["res_tv_lvms"]
            except KeyError:
                continue
            # for now
            if len(res_tv_lvms_) == 0:
                continue
            # print(subj_id, sess_id, region)
            res_tv_lvm[region] = {
                key: np.array([res_tv_lvm_[key] for res_tv_lvm_ in res_tv_lvms_])
                for key in res_tv_lvms_[0].keys()
            }
        res_tv_lvms.append(res_tv_lvm)

    return res_tv_lvms

In [ ]:
res_tv_lvms = {
    subj_id: get_res_tv_lvms(subj_id) for subj_id in ["MM012", "MR82", "MR83"]
}

In [ ]:
region = "ACC"
metric = "r2test_diff"

fig, ax = plt.subplots()
for subj_id in ["MM012", "MR82", "MR83"]:
    metrics_avg = np.array(
        [
            res_tv_lvm[region][metric].mean() if region in res_tv_lvm.keys() else np.nan
            for res_tv_lvm in res_tv_lvms[subj_id]
        ]
    )
    metrics_std = np.array(
        [
            res_tv_lvm[region][metric].std() if region in res_tv_lvm.keys() else np.nan
            for res_tv_lvm in res_tv_lvms[subj_id]
        ]
    )

    ax.plot(metrics_avg, color=colors_subj[subj_id], label=subj_id)
    ax.fill_between(
        np.arange(len(metrics_avg)),
        metrics_avg - metrics_std,
        metrics_avg + metrics_std,
        color=colors_subj[subj_id],
        alpha=0.25,
    )

ax.axhline(y=0, color="#666666", linestyle="--")
ax.set_xlabel("session")
ax.set_ylabel(metric)
ax.legend()
fig.tight_layout()


save_dir = FIGURES_DIR / "session_summary"
fpath = save_dir / f"{metric}_{region}.png"
fpath.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(fpath, dpi=300, bbox_inches="tight")

## Encoder - two strategies

In [ ]:
def load_family_strategy(subj_id, sess_id, region, seed):
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "separate_strategy"
        / "encoder_no_update_cid"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        return

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)

    family_mb = res_dict[region]["mb"]["families"][seed]
    family_mf = res_dict[region]["mf"]["families"][seed]

    return family_mb, family_mf

In [ ]:
from sg.fitter import LVMFamily

subj_id = "MR82"
sess_id = "20251027_152036"
reg = "DLS"
seed = 0

family = LVMFamily(
    subj_id=subj_id,
    sess_id=sess_id,
    n_latents_mult=1,
    n_latents_addt=1,
    sanity_check=0,
    task_vars=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    n_splines=5,
    tpre=0.5,
    tpost=1,
)
family.fit_all()
family.eval()

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond, get_choice_ts
from utils.paths import FIGURES_DIR
from pathlib import Path


def plot_psths_renderers(family, mode, reg, strategy):
    renderer = PETHRasterRenderer(
        event_times=get_choice_ts(family.trial_data, mode=mode),
        spike_times=family.spike_times[reg],
        peths=get_psths_cond(family.psths[reg], family.trial_data, mode=mode),
        pres=0.5,
        posts=1,
        binwidth_s=25 / 1000,
        s=0.5,
        linewidths=0.5,
        save_subdir=Path("peths") / subj_id / sess_id / reg / mode / strategy,
    )

    _ = NeuronViewer(
        num_units=family.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
    )

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"
reg = "DLS"
seed = 0

family_mb, family_mf = load_family_strategy(subj_id, sess_id, reg, seed)

In [ ]:
# try ridge and look at weights

In [ ]:
plot_psths_renderers(family, "response", reg, "all")

In [ ]:
plot_psths_renderers(family_mb, "response", reg, "mb")

In [ ]:
plot_psths_renderers(family_mf, "response", reg, "mf")

In [ ]:
# find regl constant fit to full model then fit mb and mf separately on different epochs

In [ ]:
def plot_beta_tv(subj_id, sess_id, tv, region, seed, color=False):
    family_mb, family_mf = load_family_strategy(subj_id, sess_id, region, seed)

    beta_mb = family_mb.mod_taskvar.tv.weight.data[:]
    beta_mf = family_mf.mod_taskvar.tv.weight.data[:]

    tv_idxs = []
    tv_labels = []
    counter = 0
    for tv_ in family_mb.task_vars:
        for val in family_mb.trial_data[tv_].unique():
            if tv_ == tv:
                tv_idxs.append(counter)
                tv_labels.append(f"{tv_}_{val}")
            counter += 1

    fig, axes = plt.subplots(nrows=1, ncols=len(tv_idxs))

    for i, ax in enumerate(axes.flat):
        if color:
            c = ax.scatter(
                beta_mb[tv_idxs[i]],
                beta_mf[tv_idxs[i]],
                s=0.5,
                cmap="hsv",
                c=np.arange(beta_mb.shape[1]),
            )
            fig.colorbar(c)
        else:
            ax.scatter(beta_mb[tv_idxs[i]], beta_mf[tv_idxs[i]], s=0.5, color="#17612F")
        ax.set_xlabel(r"mb $\beta$")
        ax.set_ylabel(r"mf $\beta$")
        ax.set_title(f"{tv_labels[i]}")
    fig.tight_layout()


plot_beta_tv(subj_id, sess_id, "response", reg, seed, color=False)

## LVM - two strategies, one latent

In [ ]:
import numpy as np


def get_res_tv_lvms_strategy(subj_id, regions=["all", "ACC", "M2", "DMS", "DLS"]):
    subj_idx = np.where(subject_ids == subj_id)[0][0]

    res_tv_lvms = []
    n_trials = []
    for sess_id in session_ids[subj_idx]:
        file_path = (
            MODELS_DIR
            / "fit"
            / subj_id
            / sess_id
            / "separate_strategy"
            / "results_dict.pkl"
        )

        if not file_path.is_file():
            continue

        with open(file_path, "rb") as f:
            res_dict = pickle.load(f)

        res_tv_lvm = {"mb": {}, "mf": {}}
        try:
            n_trials.append(res_dict["all"]["mb"]["families"][0].num_trials)
        except KeyError:
            continue

        for region in regions:
            try:
                res_tv_lvms_mb_ = res_dict[region]["mb"]["res_tv_lvms"]
                res_tv_lvms_mf_ = res_dict[region]["mf"]["res_tv_lvms"]
            except KeyError:
                continue
            # for now
            if len(res_tv_lvms_mb_) == 0 or len(res_tv_lvms_mf_) == 0:
                continue
            print(subj_id, sess_id, region)
            res_tv_lvm["mb"][region] = {
                key: np.array([res_tv_lvm_[key] for res_tv_lvm_ in res_tv_lvms_mb_])
                for key in res_tv_lvms_mb_[0].keys()
            }
            res_tv_lvm["mf"][region] = {
                key: np.array([res_tv_lvm_[key] for res_tv_lvm_ in res_tv_lvms_mf_])
                for key in res_tv_lvms_mf_[0].keys()
            }
        res_tv_lvms.append(res_tv_lvm)

    return res_tv_lvms, n_trials

In [ ]:
out = {
    subj_id: get_res_tv_lvms_strategy(subj_id) for subj_id in ["MM012", "MR82", "MR83"]
}

res_tv_lvms_strategy = {
    subj_id: out[subj_id][0] for subj_id in ["MM012", "MR82", "MR83"]
}
n_trials = {subj_id: out[subj_id][1] for subj_id in ["MM012", "MR82", "MR83"]}

In [ ]:
from core.data import colors_strategy

region = "all"
metric = "r2test_affine"
subj_id = "MR83"

fig, ax = plt.subplots()

metrics_avg_mb = np.array(
    [
        res_tv_lvm["mb"][region][metric].mean()
        if region in res_tv_lvm["mb"].keys()
        else np.nan
        for res_tv_lvm in res_tv_lvms_strategy[subj_id]
    ]
)
metrics_std_mb = np.array(
    [
        res_tv_lvm["mb"][region][metric].std()
        if region in res_tv_lvm["mb"].keys()
        else np.nan
        for res_tv_lvm in res_tv_lvms_strategy[subj_id]
    ]
)

metrics_avg_mf = np.array(
    [
        res_tv_lvm["mf"][region][metric].mean()
        if region in res_tv_lvm["mf"].keys()
        else np.nan
        for res_tv_lvm in res_tv_lvms_strategy[subj_id]
    ]
)
metrics_std_mf = np.array(
    [
        res_tv_lvm["mf"][region][metric].std()
        if region in res_tv_lvm["mf"].keys()
        else np.nan
        for res_tv_lvm in res_tv_lvms_strategy[subj_id]
    ]
)

ax.plot(metrics_avg_mb, color=colors_strategy["mb"], label="mb")
ax.plot(metrics_avg_mf, color=colors_strategy["mf"], label="mf")
ax.fill_between(
    np.arange(len(metrics_avg_mb)),
    metrics_avg_mb - metrics_std_mb,
    metrics_avg_mb + metrics_std_mb,
    color=colors_strategy["mb"],
    alpha=0.25,
)
ax.fill_between(
    np.arange(len(metrics_avg_mf)),
    metrics_avg_mf - metrics_std_mf,
    metrics_avg_mf + metrics_std_mf,
    color=colors_strategy["mf"],
    alpha=0.25,
)

ax2 = ax.twinx()
ax2.plot(n_trials[subj_id], color="#888888")
ax2.set_ylabel("n. trials")

ax.axhline(y=0, color="#666666", linestyle="--")
ax.set_xlabel("session")
ax.set_ylabel(metric)
ax.legend()
fig.tight_layout()


save_dir = FIGURES_DIR / "session_summary" / "separate_strategy"
fpath = save_dir / f"{metric}_{region}.png"
fpath.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(fpath, dpi=300, bbox_inches="tight")

In [ ]:
def load_family_strategy(subj_id, sess_id, region, seed):
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "separate_strategy"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        return

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)

    family_mb = res_dict[region]["mb"]["families"][seed]
    family_mf = res_dict[region]["mf"]["families"][seed]

    return family_mb, family_mf

In [ ]:
family_mb, family_mf = load_family_strategy("MR82", "20251027_152036", "all", 0)

In [ ]:
(
    family_mb.mod_taskvar.tv.weight.data[:].shape,
    family_mf.mod_taskvar.tv.weight.data[:].shape,
)

In [ ]:
from sg.eval_models import get_coupling


def plot_coupling_strategy(subj_id, sess_id, region, seed):
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "separate_strategy"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        return

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)

    family_mb = res_dict[region]["mb"]["families"][seed]
    family_mf = res_dict[region]["mf"]["families"][seed]

    fig, axes = plt.subplots(nrows=2, ncols=2)

    # gain
    coupling_gg_mb = get_coupling(family_mb.mod_gain, mode="gain")
    coupling_gg_mf = get_coupling(family_mf.mod_gain, mode="gain")

    # offset
    coupling_oo_mb = get_coupling(family_mb.mod_offset, mode="offset")
    coupling_oo_mf = get_coupling(family_mf.mod_offset, mode="offset")

    # affine
    coupling_ga_mb = get_coupling(family_mb.mod_affine, mode="gain")
    coupling_oa_mb = get_coupling(family_mb.mod_affine, mode="offset")
    coupling_ga_mf = get_coupling(family_mf.mod_affine, mode="gain")
    coupling_oa_mf = get_coupling(family_mf.mod_affine, mode="offset")

    # PLOT
    print(coupling_gg_mb.shape, coupling_gg_mf.shape)
    axes[0, 0].scatter(coupling_gg_mb, coupling_gg_mf)
    axes[0, 1].scatter(coupling_oo_mb, coupling_oo_mf)
    axes[1, 0].scatter(coupling_ga_mb, coupling_ga_mf)
    axes[1, 1].scatter(coupling_oa_mb, coupling_oa_mf)

In [ ]:
plot_coupling_strategy("MR82", "20251027_152036", "all", 0)